# Task 1 / OMD: 本文チャンク＋事実一致特徴を用いた Fine-tuning

引用文脈と引用先論文のペアを ModernBERT で判定する cross-encoder に、特徴量実験で有効だった
**事実一致特徴**を追加したハイブリッドモデルです。

## このNotebookの手法

1. **本文チャンク検索**: 引用文と意味的に近い論文本文の上位3チャンクを取得し、モデルへの入力に加える。
2. **事実一致特徴（中心となる改善）**: 引用文に出る数値・年・英字の手法名・著者名・日本語専門語が、論文情報にも出現する割合を数値化する。
3. **言い換え・話題の近さ**: LSAで圧縮した意味空間における、引用文とタイトル・概要の類似度を数値化する。
4. **引用文の形式**: 引用位置、文の長さ、数値数、主張表現・課題表現の数を数値化する。
5. **ハイブリッド分類**: ModernBERTの文ペア表現と上記の数値特徴を結合し、妥当（1）／不適切（0）を判定する。

特徴量アブレーションでは、単純な文字n-gramの表層類似度は効果が安定しなかったため、**表層類似度は数値特徴として追加しません**。
ただし、論文のタイトル・概要・本文チャンクそのものは、cross-encoderが内容を読むための入力として使います。

**使用モデル**: 分類用 `sbintuitions/modernbert-ja-70m`（70M）と、本文検索用 `cl-nagoya/ruri-v3-30m`（30M）。どちらも規定の1モデル100M以下です。

In [ ]:
# 必要なライブラリ（初回のみ）
%pip install -q transformers sentencepiece sentence-transformers scikit-learn

In [ ]:
# データの取得（Google Colab 用）: 配布リポジトリをクローンする。
# ローカルで配布リポジトリの中から実行している場合、このセルは何もしません。
![ -d data ] || [ -d ../data ] || git clone -q https://github.com/YANS-official/yans-2026-hackathon

In [ ]:
import json
import os
import random
from pathlib import Path

import numpy as np
import torch

# 再現性のためにシードを固定
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# 配布データの場所を自動で探す（./data → ../data → クローン先）
_candidates = ([Path(os.environ["YANS_DATA_DIR"])] if os.environ.get("YANS_DATA_DIR") else []) + [
    Path("data"), Path("../data"), Path("yans-2026-hackathon/data")]
DATA_DIR = next((p for p in _candidates if (p / "train.jsonl").exists()), None)
assert DATA_DIR is not None, "配布データ（data/）が見つかりません"
print(f"データディレクトリ: {DATA_DIR.resolve()}")

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train = read_jsonl(DATA_DIR / "train.jsonl")
dev = read_jsonl(DATA_DIR / "dev_labeled.jsonl")
lb = read_jsonl(DATA_DIR / "dev_leaderboard.jsonl")
papers = {p["paper_id"]: p for p in read_jsonl(DATA_DIR / "papers.jsonl")}
print(f"train={len(train)}  dev_labeled={len(dev)}  dev_leaderboard={len(lb)}")

## 1. 本文検索・補助特徴・モデルの準備

文ペア分類器だけでは、数値や著者名などの一致を明示的に扱いにくいため、引用文と論文情報から作った補助特徴を追加します。
これらは **trainのみでfitしたLSA・標準化器**で処理し、devやリーダーボードのラベルは一切使いません。

### 1.1 本文チャンクの検索

引用文脈と意味的に近い論文本文の上位3チャンクを検索します。検索結果はcross-encoderの入力文Bに追加します。
本文類似度そのものを単独の数値特徴として強く使うのではなく、モデルが根拠本文を直接読めるようにする設計です。

In [ ]:
%pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

CHUNK_MODEL_NAME = "cl-nagoya/ruri-v3-30m"
TOP_N_CHUNKS = 3

chunks_by_paper = {}
with open(DATA_DIR / "chunks.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        chunks_by_paper.setdefault(rec["paper_id"], []).append(rec["text"])

chunk_model = SentenceTransformer(CHUNK_MODEL_NAME, device=device)
print("チャンク検索用モデル パラメータ数:",
      f"{sum(p.numel() for p in chunk_model.parameters())/1e6:.0f}M")

# 論文ごとにチャンクを埋め込む（引用文脈との類似度検索に使う）
chunk_embs_by_paper = {}
for pid, texts in chunks_by_paper.items():
    chunk_embs_by_paper[pid] = chunk_model.encode(texts, normalize_embeddings=True)

# 全レコードの引用文脈をまとめて埋め込み、各レコードについて対象論文のチャンクの中から
# 類似度上位 TOP_N_CHUNKS 件を検索してテキストに連結しておく（学習中に毎回埋め込み直さないよう
# 事前計算してキャッシュする）
def build_top_chunks_cache(records):
    contexts = [r["citation_context"] for r in records]
    ctx_embs = chunk_model.encode(contexts, normalize_embeddings=True, show_progress_bar=True, batch_size=64)
    cache = {}
    for r, ctx_emb in zip(records, ctx_embs):
        pid = r["cited_paper_id"]
        texts = chunks_by_paper.get(pid)
        embs = chunk_embs_by_paper.get(pid)
        if not texts or embs is None or len(embs) == 0:
            cache[r["id"]] = ""
            continue
        sims = embs @ ctx_emb
        top_idx = np.argsort(-sims)[:TOP_N_CHUNKS]
        cache[r["id"]] = " ".join(texts[i] for i in top_idx)
    return cache

top_chunks_cache = {}
for name, records in [("train", train), ("dev", dev), ("lb", lb)]:
    print(f"チャンク検索キャッシュを構築中: {name} ({len(records)}件) ...")
    top_chunks_cache.update(build_top_chunks_cache(records))

In [ ]:
import re
from functools import lru_cache

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from transformers import AutoModel, AutoTokenizer

# ----- 特徴量実験で有望だった3群: 事実一致・言い換え類似・引用文形式 -----
CITE_RE = re.compile(r"\[CITE\]")
NUM_RE = re.compile(r"(?<![A-Za-z])\d+(?:[.,]\d+)?%?")
YEAR_RE = re.compile(r"(?:19|20)\d{2}")
LATIN_RE = re.compile(r"[A-Za-z][A-Za-z0-9+_.-]{1,}")
JP_TOKEN_RE = re.compile(r"[一-龥ァ-ヴー]{2,}")
CLAIM_CUES = ["提案", "示した", "報告", "明らか", "確認", "達成", "向上", "低下", "有効", "可能"]
NEG_CUES = ["一方", "しかし", "課題", "問題", "限界", "ない", "ず", "未"]

FEATURE_NAMES = [
    "数値一致率", "年一致率", "英字語一致率", "著者名一致", "専門語一致率",
    "LSAタイトル類似度", "LSA概要類似度", "LSAタイトル概要類似度",
    "引用文長", "引用位置", "数値数", "主張表現数", "課題表現数",
]

def clean_text(text):
    return re.sub(r"\s+", " ", CITE_RE.sub(" ", text or "")).strip()

def recall_in_text(items, target):
    items = {x.lower() for x in items}
    if not items:
        return 0.0
    target = target.lower()
    return sum(item in target for item in items) / len(items)

# LSAはtrainの文脈・引用先論文情報だけでfitする（dev/lbのラベルは用いない）。
train_pids = sorted({r["cited_paper_id"] for r in train})
lsa_fit_texts = [clean_text(r["citation_context"]) for r in train]
lsa_fit_texts += [clean_text(papers[pid].get("title", "") + " " + papers[pid].get("abstract", ""))
                  for pid in train_pids]
lsa_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2,
                            max_features=45000, sublinear_tf=True, norm="l2")
lsa_tfidf.fit(lsa_fit_texts)
lsa_dim = min(128, len(train) - 1, len(lsa_tfidf.get_feature_names_out()) - 1)
lsa_model = TruncatedSVD(n_components=lsa_dim, n_iter=7, random_state=SEED)
lsa_model.fit(lsa_tfidf.transform(lsa_fit_texts))

def cosine_dense(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0

def feature_vector(rec):
    """表層文字列類似度は入れず、事実一致・LSA・形式特徴だけを返す。"""
    ctx_raw = rec["citation_context"]
    ctx = clean_text(ctx_raw)
    p = papers[rec["cited_paper_id"]]
    title = clean_text(p.get("title", ""))
    abstract = clean_text(p.get("abstract", ""))
    evidence = clean_text(top_chunks_cache.get(rec["id"], ""))
    paper_text = " ".join([title, abstract, evidence])
    author_tokens = [x.strip() for x in re.split(r"\s+|\band\b|,|・", p.get("author", "")) if len(x.strip()) >= 2]

    q_lsa = lsa_model.transform(lsa_tfidf.transform([ctx]))[0]
    paper_lsa = lsa_model.transform(lsa_tfidf.transform([title, abstract, title + " " + abstract]))
    cite_pos = ctx_raw.find("[CITE]") / max(1, len(ctx_raw))

    return np.asarray([
        recall_in_text(NUM_RE.findall(ctx), paper_text),
        recall_in_text(YEAR_RE.findall(ctx), paper_text),
        recall_in_text(LATIN_RE.findall(ctx), paper_text),
        float(any(token.lower() in ctx.lower() for token in author_tokens)),
        recall_in_text(JP_TOKEN_RE.findall(ctx), title + " " + abstract),
        cosine_dense(q_lsa, paper_lsa[0]),
        cosine_dense(q_lsa, paper_lsa[1]),
        cosine_dense(q_lsa, paper_lsa[2]),
        np.log1p(len(ctx)), cite_pos,
        np.log1p(len(NUM_RE.findall(ctx))),
        sum(ctx.count(cue) for cue in CLAIM_CUES),
        sum(ctx.count(cue) for cue in NEG_CUES),
    ], dtype=np.float32)

all_records = train + dev + lb
raw_features_by_id = {r["id"]: feature_vector(r) for r in all_records}
feature_scaler = StandardScaler().fit(np.stack([raw_features_by_id[r["id"]] for r in train]))
features_by_id = {rid: feature_scaler.transform(vec[None])[0].astype(np.float32)
                  for rid, vec in raw_features_by_id.items()}
print(f"補助特徴: {len(FEATURE_NAMES)}個")
print(FEATURE_NAMES)

# ----- cross-encoder本体 -----
MODEL_NAME = "sbintuitions/modernbert-ja-70m"
MAX_LEN = 320
BATCH_SIZE = 16
EPOCHS = 5
LR = 3e-5
TOP_N_CHUNKS = 3

if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
    print("警告: GPU がありません。ColabではT4 GPUを選んでください。")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode_batch(records):
    contexts = [r["citation_context"] for r in records]
    paper_texts = []
    for r in records:
        p = papers[r["cited_paper_id"]]
        evidence = top_chunks_cache.get(r["id"], "")
        paper_texts.append(f"論文タイトル: {p['title']}\n概要: {p['abstract']}\n本文根拠: {evidence}")
    enc = tokenizer(contexts, paper_texts, truncation=True, max_length=MAX_LEN,
                    padding=True, return_tensors="pt")
    hand_features = torch.tensor(np.stack([features_by_id[r["id"]] for r in records]), dtype=torch.float32)
    return enc, hand_features

class HybridCitationModel(torch.nn.Module):
    """ModernBERTの[CLS]表現と、事実一致等の13特徴を結合する分類器。"""
    def __init__(self, model_name, n_features):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.classifier = torch.nn.Sequential(
            torch.nn.Dropout(0.15),
            torch.nn.Linear(hidden + n_features, 256),
            torch.nn.GELU(),
            torch.nn.Dropout(0.15),
            torch.nn.Linear(256, 2),
        )

    def forward(self, hand_features, **encoder_inputs):
        output = self.encoder(**encoder_inputs)
        cls = output.last_hidden_state[:, 0]
        logits = self.classifier(torch.cat([cls, hand_features], dim=1))
        return logits

def build_model():
    hybrid = HybridCitationModel(MODEL_NAME, len(FEATURE_NAMES)).to(device)
    n_params = sum(p.numel() for p in hybrid.parameters()) / 1e6
    print(f"分類モデルのパラメータ数: {n_params:.1f}M（100M以下）")
    return hybrid

model = build_model()

## 2. ハイブリッドモデルの Fine-tuning

ModernBERTが引用文・タイトル・概要・本文根拠の内容を読み、分類ヘッドがそれに事実一致・LSA類似度・引用文形式の13個の数値特徴を結合して判定します。
各エポック後にdev_labeledで評価し、Accuracyが最も高い重みを採用します。

In [ ]:
import copy

from sklearn.metrics import accuracy_score, f1_score
from transformers import get_linear_schedule_with_warmup

# 再実行しても前回の学習済み重みを引き継がないよう毎回初期化する。
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
model = build_model()

@torch.no_grad()
def predict(records, batch_size=64, return_probability=False):
    model.eval()
    pred_list, prob_list = [], []
    for start in range(0, len(records), batch_size):
        enc, hand_features = encode_batch(records[start:start + batch_size])
        enc = {key: value.to(device) for key, value in enc.items()}
        logits = model(hand_features=hand_features.to(device), **enc)
        prob = logits.softmax(dim=-1)[:, 1].cpu().numpy()
        prob_list.extend(prob)
        pred_list.extend((prob >= 0.5).astype(int))
    if return_probability:
        return np.asarray(pred_list), np.asarray(prob_list)
    return np.asarray(pred_list)

y_train = np.array([int(r["label"]) for r in train])
y_dev = np.array([int(r["label"]) for r in dev])
order = list(range(len(train)))
steps_per_epoch = (len(train) + BATCH_SIZE - 1) // BATCH_SIZE
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(0.1 * EPOCHS * steps_per_epoch), EPOCHS * steps_per_epoch)

checkpoints = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    random.shuffle(order)
    total_loss = 0.0
    for start in range(0, len(order), BATCH_SIZE):
        batch = [train[idx] for idx in order[start:start + BATCH_SIZE]]
        enc, hand_features = encode_batch(batch)
        enc = {key: value.to(device) for key, value in enc.items()}
        labels = torch.tensor([int(r["label"]) for r in batch], device=device)
        logits = model(hand_features=hand_features.to(device), **enc)
        loss = torch.nn.functional.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        total_loss += float(loss)
    dev_pred = predict(dev)
    acc = accuracy_score(y_dev, dev_pred)
    print(f"epoch {epoch}: 平均loss={total_loss / steps_per_epoch:.3f}  dev Accuracy={acc:.3f}")
    checkpoints.append((acc, copy.deepcopy(model.state_dict())))

adopt_acc, adopt_state = max(checkpoints, key=lambda x: x[0])
model.load_state_dict(adopt_state)
print(f"\n採用: dev Accuracy={adopt_acc:.3f} のエポックの重み")

## 3. 性能と特徴の確認

まずfine-tuningの性能を確認します。次に、誤り事例について、数値・年・著者名・専門語などの事実一致がどの程度あったかを確認します。

In [ ]:
pred_train = predict(train)
pred_dev, prob_dev = predict(dev, return_probability=True)
print(f"[ハイブリッドfine-tuning] train      : Accuracy={accuracy_score(y_train, pred_train):.3f}  "
      f"F1={f1_score(y_train, pred_train):.3f}")
print(f"[ハイブリッドfine-tuning] dev_labeled: Accuracy={accuracy_score(y_dev, pred_dev):.3f}  "
      f"F1={f1_score(y_dev, pred_dev):.3f}")

def show_cases(records, golds, preds, probs, correct, n=3):
    shown = 0
    for i, r in enumerate(records):
        if (preds[i] == golds[i]) != correct:
            continue
        p = papers[r["cited_paper_id"]]
        feat = dict(zip(FEATURE_NAMES, raw_features_by_id[r["id"]]))
        fact_view = {k: round(float(feat[k]), 3) for k in FEATURE_NAMES[:5]}
        print(f"--- {r['id']}  正解={golds[i]}  予測={preds[i]}  P(妥当)={probs[i]:.3f} ---")
        print("引用文:", r["citation_context"])
        print("論文:", p["title"], f"（{p['year']}）")
        print("事実一致特徴:", fact_view)
        print()
        shown += 1
        if shown >= n:
            return

print("========== 不正解の事例 ==========")
show_cases(dev, y_dev, pred_dev, prob_dev, correct=False, n=5)

## 4. 最終出力

選択された最良エポックのモデルで `dev_leaderboard` の予測を作ります。最終フェーズでは、データ読み込みセルの `lb` を `test.jsonl` に置き換えて同じ処理を実行できます。

In [ ]:
pred_lb = predict(lb)
output_name = "submission_task1_ft_factmatch_chunks.jsonl"
with open(output_name, "w", encoding="utf-8") as f:
    for rec, pred in zip(lb, pred_lb):
        f.write(json.dumps({"id": rec["id"], "prediction": int(pred)}, ensure_ascii=False) + "\n")
print(f"{output_name} を書き出しました（{len(lb)}行 / 予測ラベル分布: "
      f"1が{int(pred_lb.sum())}問, 0が{len(lb) - int(pred_lb.sum())}問）")

## 5. この実験で確認すること

1. まずこのハイブリッドモデルのdev Accuracy / F1を記録する。
2. `task1_finetune_chunks.ipynb` の旧版（本文チャンクのみ）と比較し、事実一致特徴の追加が改善するかを確認する。
3. 事実一致特徴が有効だった特徴量実験の結果と、fine-tuningでも同じ傾向が出るかを誤り事例で確認する。
4. devに何度も合わせすぎないよう、乱数seedを替えた再実行でも改善が続くか確認する。

**提出前の確認**: 分類モデル約70M、本文検索モデル30Mであり、それぞれ100M以下です。学習・推論は配布データのみを用い、リーダーボードやtestのラベル推定を学習には使いません。